# Process Intake Checklist

Reads the authoritative POC focus-area data extracted from
`Reliability-Intake-Checklist.Rev0.xlsx` (uploaded to `Files/intake-checklist/`)
and persists it as Delta tables in the `gold` schema.

| Output table | Source CSV | Purpose |
|---|---|---|
| `gold.dim_scope_asset` | `in_scope_assets.csv` | 3 MUST-HAVE assets in scope for the POC |
| `gold.bridge_pi_tag_to_asset` | `sensor_tags.csv` | **Authoritative** PI tag → asset_id (308 rows, HIGH/MED/LOW relevance) |
| `gold.bridge_gads_event_to_asset` | `downtime_history.csv` (GADRS rows) | **Authoritative** GADS `GEN_SEQ_NO` → asset_id |
| `gold.running_indicator` | `running_indicators.csv` | Per-asset "asset is running" tag + threshold |
| `gold.downtime_history` | `downtime_history.csv` (all rows) | Full downtime history reference table |

These tables are consumed by `Build-Gold-Model` to dramatically lift PI / GADS coverage
inside the POC focus area.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
BASE = "Files/intake-checklist"
print("Reading from", BASE)

## 1. `gold.dim_scope_asset`

3 MUST-HAVE assets. `asset_id` is the primary key for all intake-checklist joins.

In [ ]:
df_scope = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{BASE}/in_scope_assets.csv"))

df_scope = (df_scope
    .withColumnRenamed("criticality (H/M/L)", "criticality")
    .withColumnRenamed("approx_unplanned_hours_last_12mo", "approx_unplanned_hours_last_12mo")
    .withColumn("approx_unplanned_hours_last_12mo",
                F.col("approx_unplanned_hours_last_12mo").cast("double"))
    .withColumn("unit_description", F.trim("unit_description"))
    .select("asset_id", "plant", "unit_description", "criticality",
            "approx_unplanned_hours_last_12mo", "notes")
)

(df_scope.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_scope_asset"))

print(f"gold.dim_scope_asset: {spark.table('gold.dim_scope_asset').count()} rows")
spark.table("gold.dim_scope_asset").show(truncate=False)

## 2. `gold.bridge_pi_tag_to_asset`

**Authoritative** PI tag → asset_id, classified by downtime relevance (HIGH / MEDIUM / LOW).
This replaces the inferred `bridge_pi_tag_to_equipment` for in-scope PI tags.

In [ ]:
df_tags = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{BASE}/sensor_tags.csv"))

# Normalise / clean
df_tags = (df_tags
    .withColumn("tag_name", F.trim("tag_name"))
    .withColumn("asset_id", F.trim("asset_id"))
    .withColumn("downtime_relevance", F.upper(F.trim("downtime_relevance")))
    .filter(F.col("tag_name").isNotNull() & (F.col("tag_name") != ""))
    .select(
        F.col("tag_name").alias("Tag"),      # match column name used in pi_data
        "asset_id",
        "tag_role",
        "downtime_relevance",
        F.col("engineering_units").alias("eng_units"),
        F.col("description").alias("tag_description"),
        "notes",
    )
)

# Sanity-check: no duplicate Tag values (the bridge must be 1:1)
dups = df_tags.groupBy("Tag").count().filter("count > 1").count()
print(f"Duplicate Tag rows: {dups}")

(df_tags.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_pi_tag_to_asset"))

total = spark.table("gold.bridge_pi_tag_to_asset").count()
print(f"gold.bridge_pi_tag_to_asset: {total} rows")

(spark.table("gold.bridge_pi_tag_to_asset")
    .groupBy("asset_id", "downtime_relevance").count()
    .orderBy("asset_id", "downtime_relevance")
    .show(truncate=False))

## 3. `gold.bridge_gads_event_to_asset`

GADRS `source_id` from the downtime history equals `GEN_SEQ_NO` on `gads_events_enriched`.
This gives us an authoritative GADS event → asset link for the POC scope.

In [ ]:
df_dt = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{BASE}/downtime_history.csv"))

# Just the GADRS rows have GEN_SEQ_NO mappings
df_bridge = (df_dt
    .filter(F.upper(F.trim("source_system")) == "GADRS")
    .withColumn("gen_seq_no", F.col("source_id").cast("long"))
    .filter(F.col("gen_seq_no").isNotNull())
    .select(
        "gen_seq_no",
        F.trim("asset_id").alias("asset_id"),
        F.trim("classification").alias("downtime_classification"),
        F.trim("root_cause_summary").alias("root_cause_summary"),
        F.trim("notes").alias("gads_intake_notes"),
    )
)

(df_bridge.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.bridge_gads_event_to_asset"))

total = spark.table("gold.bridge_gads_event_to_asset").count()
print(f"gold.bridge_gads_event_to_asset: {total} rows")
spark.table("gold.bridge_gads_event_to_asset").orderBy("asset_id", "gen_seq_no").show(truncate=False)

## 4. `gold.running_indicator`

Per-asset tag + threshold defining "asset is running" — useful for distinguishing
real downtime from planned shutdowns when analysing PI data.

In [ ]:
df_run = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{BASE}/running_indicators.csv"))

df_run = (df_run
    .withColumnRenamed("operator (>, >=, <, <=, ==)", "operator")
    .withColumn("threshold_value", F.col("threshold_value").cast("double"))
    .select(
        F.trim("asset_id").alias("asset_id"),
        F.trim("running_tag_name").alias("Tag"),
        "operator", "threshold_value",
        F.trim("units").alias("units"),
        F.trim("tag_description").alias("tag_description"),
        "notes",
    )
    .filter(F.col("Tag").isNotNull() & (F.col("Tag") != ""))
)

(df_run.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.running_indicator"))

print(f"gold.running_indicator: {spark.table('gold.running_indicator').count()} rows")
spark.table("gold.running_indicator").show(truncate=False)

## 5. `gold.downtime_history`

Full downtime history reference table — every row from sheet 3 regardless of source system.
Useful for analyst drill-down from any fact row.

In [ ]:
df_full = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(f"{BASE}/downtime_history.csv"))

df_full = (df_full
    .withColumn("event_start_ts", F.to_timestamp("event_start", "M/d/yy H:mm"))
    .withColumn("event_end_ts",   F.to_timestamp("event_end",   "M/d/yy H:mm"))
    .select(
        F.trim("asset_id").alias("asset_id"),
        "event_start", "event_end",
        "event_start_ts", "event_end_ts",
        F.trim("classification").alias("classification"),
        "root_cause_summary",
        F.upper(F.trim("source_system")).alias("source_system"),
        F.trim("source_id").alias("source_id"),
        "notes",
    )
)

(df_full.write.mode("overwrite").format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.downtime_history"))

print(f"gold.downtime_history: {spark.table('gold.downtime_history').count()} rows")
(spark.table("gold.downtime_history")
    .groupBy("asset_id", "source_system").count()
    .orderBy("asset_id", "source_system")
    .show(truncate=False))

## 6. Coverage cross-check

Before `Build-Gold-Model` re-runs, verify how many PI value rows and GADS events
the authoritative bridges actually cover.

In [ ]:
# PI coverage with authoritative bridge
pi_total  = spark.table("pi_data").count()
pi_in_br  = spark.table("pi_data").join(
    spark.table("gold.bridge_pi_tag_to_asset").select("Tag"), "Tag"
).count()
pi_pct = 100.0 * pi_in_br / pi_total if pi_total else 0.0
print(f"PI rows covered by authoritative bridge: {pi_in_br:,} / {pi_total:,}  ({pi_pct:.1f}%)")

# GADS coverage with authoritative bridge
gads_total = (spark.table("gads_events_enriched")
    .filter(F.col("UNIT_ID").isin(86, 87)).count())
gads_in_br = (spark.table("gads_events_enriched")
    .filter(F.col("UNIT_ID").isin(86, 87))
    .join(spark.table("gold.bridge_gads_event_to_asset")
            .select(F.col("gen_seq_no").alias("GEN_SEQ_NO")),
          "GEN_SEQ_NO")
    .count())
gads_pct = 100.0 * gads_in_br / gads_total if gads_total else 0.0
print(f"GADS events covered by authoritative bridge: {gads_in_br:,} / {gads_total:,}  ({gads_pct:.1f}%)")

# Distinct PI tag count per asset
print("\nDistinct PI tag count per scope asset (authoritative bridge):")
(spark.table("gold.bridge_pi_tag_to_asset")
    .groupBy("asset_id").agg(F.countDistinct("Tag").alias("tags"))
    .orderBy("asset_id").show(truncate=False))